In [ ]:
from src.run_app import *
from src.prepare_gam import *
from src.utils import *
from gam_rs_utils.binarize_dataset import binarize_dataset
from gam_rs_utils.utils import *
from FasterRisk.src.fasterrisk import fasterrisk
from time import time

dataset_settings = {
    'bank': {
        'gap_tolerance': 0.004,
        'starting_num_estimators': 50,
    },
    'compas': {
        'gap_tolerance': 0.003,
        'starting_num_estimators': 50,
    },
    'diabetes': {
        'gap_tolerance': 0.006,
        'starting_num_estimators': 200,
    },
    'heloc_original': {
        'gap_tolerance': 0.003,
        'starting_num_estimators': 50,
    },
    # 'hiv': { # not used
    #     'gap_tolerance': 0.001,
    #     'starting_num_estimators': 25,
    # },
    'netherlands': {
        'gap_tolerance': 0.001,
        'starting_num_estimators': 100,
    },
    'spambase': {
        'gap_tolerance': 0.005,
        'starting_num_estimators': 25,
    },
}

tries = 6
for dataset_name, settings in dataset_settings.items():
    gap_tolerance = settings['gap_tolerance']

    path = 'datasets/{}.csv'.format(dataset_name)
    dataset = pd.read_csv(path)
    print(f"Dataset: {dataset_name}")
    print(f"shape: {dataset.shape}")

    results = []
    for i in range(1, tries + 1):
        num_estimators = settings["starting_num_estimators"] * i

        df, thresholds, header, threshold_guess_time = binarize_dataset(dataset, num_estimators)
        X, y = df.iloc[:, :-1], df.iloc[:, -1]

        header = list(X.columns)
        header = pd.Index(["intercept"] + header)
        header = header.astype("object")

        X_one_hot, y = utils.get_X_y(X, y)

        start = time()
        rs = fasterrisk.RiskScoreOptimizer(X_one_hot, y, k=10, lb=-100, ub=100, gap_tolerance=gap_tolerance, select_top_m=-1)
        rs.optimize_with_swaps(swaps=3, fanout_decay=1, feature_selection="top")
        end = time()
        
        results.append((dataset_name, len(header), end - start, rs.sparseDiversePool_betas, rs.sparseDiversePool_beta0))
        print(f"\t{len(header)} features, {rs.sparseDiversePool_betas.shape[0]} solutions, {end - start:.2f} seconds")

In [ ]:
import pickle
with open("results/features.pkl", "rb") as f:
    results = pickle.load(f)